In [12]:
import pandas as pd
from fastapi import FastAPI, HTTPException
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

app = FastAPI()

In [13]:
df_movies = pd.read_csv(r'movies.csv', dtype={'original_title': str}, low_memory=False)

In [14]:
df_movies = pd.read_csv(r'movies.csv')
df_movies.dtypes

C:\Users\anavi\AppData\Local\Temp\ipykernel_18900\3577449744.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_movies = pd.read_csv(r'movies.csv')


budget            float64
id                 object
original_title     object
popularity         object
release_date       object
revenue           float64
title              object
vote_average      float64
vote_count        float64
Collection         object
genres             object
ProdCompany_1      object
ProdCompany_2      object
ProdCompany_3      object
Character Name     object
Lead actor         object
Director           object
release_year        int64
return            float64
release_day        object
dtype: object

In [15]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

# Convertir los títulos en vectores numéricos
tfidf_matrix = tfidf_vectorizer.fit_transform(df_movies['original_title'].fillna(''))

# Función para encontrar películas similares
def find_similar_movies(title: str, top_n: int = 5):
    try:
        # Limpiar y procesar el título
        title_cleaned = re.sub(r'[^a-z\s]', '', title.lower())
        
        # Transformar el título ingresado en vector
        title_vector = tfidf_vectorizer.transform([title_cleaned])
        
        # Calcular similitudes coseno
        cosine_similarities = cosine_similarity(title_vector, tfidf_matrix).flatten()
        
        # Obtener índices de las películas más similares
        similar_indices = cosine_similarities.argsort()[-(top_n+1):-1][::-1]
        
        # Extraer títulos similares
        similar_titles = df_movies['original_title'].iloc[similar_indices].tolist()
        
        return similar_titles
    except Exception as e:
        raise ValueError(f"Error processing the recommendation: {str(e)}")

# Endpoint de la API
@app.get('/get_recommendation/{titulo}', response_model=list[str])
def recomendacion(titulo: str):
    try:
        # Obtener recomendaciones
        recommendations = find_similar_movies(titulo)
        if not recommendations:
            raise HTTPException(status_code=404, detail="No similar movies found.")
        return recommendations
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Internal Server Error: {str(e)}")

In [16]:
print(recomendacion('Inception'))

['Toy Story', 'Queerama', 'Satana likuyushchiy', 'Betrayal', 'Siglo ng Pagluluwal']


In [6]:
print(recomendacion('The Matrix'))

['the matrix revolutions', 'the matrix reloaded', '    ', 'how to make an american quilt', 'big bully']


In [19]:
print(recomendacion('The Shawshank Redemption'))

['na kryuchke', 'canadian bacon', '    ', 'how to make an american quilt', 'big bully']


In [20]:
print(recomendacion('Pulp Fiction'))

['stranger than fiction', 'canadian bacon', '    ', 'how to make an american quilt', 'big bully']


In [ ]:
@app.get('/get_recommendation/{titulo}', response_model=list[str])
def recomendacion(titulo: str):
    try:
        titulo = re.sub(r'[^a-z\s]', '', titulo.lower())

        titulo_vector = tfidf_vectorizer.transform([titulo])

        cosine_similarities = cosine_similarity(titulo_vector, tfidf_matrix).flatten()

        similar_indices = cosine_similarities.argsort()[-6:-1][::-1]

        similar_titles = df_movies['original_title'].iloc[similar_indices].tolist()

        return similar_titles
    except Exception as e:
        return {"error": str(e)}